# From overlapping chunks to one entry

`load_corpus` in `src/pipeline.py` joins chunk text with newlines and keeps the overlap. That join is what TF-IDF wants. Mention character offsets cannot use it: the same sentence would be counted twice, and a name split across the overlap would land at two positions.

`deoverlap` keeps the first chunk and appends only the new tail of each later chunk. `load_entries` then collapses whitespace, because the file wraps lines in the middle of a name (`Mrs.` then a newline then `Harker`). Offsets in `data/kg/mentions.jsonl` refer to that cleaned string.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "kg").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(ROOT)


In [ ]:
import json
import re

from src.kg.entries import deoverlap, load_entries, naive_join

raw = json.loads((ROOT / "data" / "dracula_ascii_rag.json").read_text(encoding="utf-8"))
rows = raw["entries"] if isinstance(raw, dict) else raw
chunks = [
    row["text"]
    for row in sorted(rows, key=lambda row: int(row.get("chunk_index") or 0))
    if int(row["entry_index"]) == 1
]
joined = naive_join(chunks)
stripped = deoverlap(chunks)
cleaned = re.sub(r"\s+", " ", stripped).strip()
print("chunks", len(chunks))
print("naive join chars", len(joined))
print("deoverlap chars", len(stripped))
print("cleaned chars", len(cleaned))
print("overlap copies of the opening", joined.count("Left Munich at 8:35"), "vs", cleaned.count("Left Munich at 8:35"))


## The entry record

Each dated diary entry, letter, telegram, memorandum or cutting becomes one `Entry`: index, chapter, narrator, dates, a document kind, and the cleaned text. An entry that opens a section before any date line carries the previous date in `date_iso`. The narrator strings `Mina Murray` and `Mina Harker` stay distinct here. Merging them is the resolver's job.


In [ ]:
entries = load_entries(ROOT / "data" / "dracula_ascii_rag.json")
first = entries[0]
print(first.entry_index, first.chapter_number, first.narrator, first.date_iso, first.kind)
print(first.text[:180])
print("entries", len(entries))
print("missing date_iso", [e.entry_index for e in entries if e.date_iso is None])
print("Mina narrator labels", sorted({e.narrator for e in entries if e.narrator and e.narrator.startswith("Mina")}))


## Where the entries come from

`data/dracula_ascii_rag.json` is built by `scripts/build_corpus.py` from the Project Gutenberg text in `data/source/pg345.txt`. An earlier version of the file had dropped sections (Mina's letter from Buda-Pesth that opens chapter IX, Van Helsing's phonograph diary that opens chapter XXIV, Jonathan's journal that opens chapter XX), put each section heading at the end of the entry before it, and gave runs of entries the chapter's first narrator. Its chapter numbers were right; chapters only looked misnumbered because their openings were missing.

`parse_book` in `src/kg/corpus.py` walks the book paragraph by paragraph. `CHAPTER IX` sets the chapter. A heading such as `_Dr. Seward's Diary._` or `_Letter, Mina Harker to Lucy Westenra._` opens a new entry with that writer. Inside a diary, a dated paragraph (`_20 August._--`) opens the next entry. Each entry keeps its heading as its first paragraph.

The cell finds two passages the earlier file got wrong.

In [ ]:
from collections import Counter

budapest = next(e for e in entries if "Buda-Pesth, 24 August" in e.text)
print(budapest.chapter_number, budapest.narrator, budapest.section_title)
renfield = next(e for e in entries if "The case of Renfield grows even more interesting" in e.text)
print(renfield.chapter_number, renfield.narrator, renfield.date_str)
print(Counter(e.narrator for e in entries).most_common())

## Exercise

Print one row per section title: the document kind, then the title. `document_kind` reads the title and the narrator. Journal, diary, letter, telegram, memorandum, newspaper, and phonograph should all appear. `Newspaper cutting` is a document kind, not a person.


In [ ]:
seen = []
for entry in entries:
    key = (entry.kind, entry.section_title)
    if key in seen:
        continue
    seen.append(key)
    print(f"{entry.kind:12} {entry.section_title}")
